# 01 — Exploratory Data Analysis (RAW data)

AQI forecasting for Lahore. This notebook analyzes the **pre-feature-engineering**
merged pollution + weather snapshot — not the Hopsworks engineered feature group.

**Prerequisite:**
```bash
python -m src.feature_pipeline raw-snapshot
```

Covers: structure check, univariate/bivariate/multivariate, ADF stationarity,
ACF/PACF, seasonal decomposition, smog-vs-normal comparison.

**Hard gate:** Feature engineering must wait until the Findings for FE section
below has been reviewed and approved.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:/Users/Tech Mehal/Desktop/aqi-predictor")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src import config

sns.set_theme(style="whitegrid")
%matplotlib inline

# Load raw merged snapshot (absolute path — ignore any old load_raw_snapshot cells)
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "aqi_raw_merged.parquet"
print("Loading:", RAW_PATH)
print("Exists:", RAW_PATH.exists())
assert RAW_PATH.exists(), f"Missing {RAW_PATH}"

df = pd.read_parquet(RAW_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
df = df.sort_values("timestamp").reset_index(drop=True)
df["month"] = df["timestamp"].dt.month
df["is_smog_season"] = df["month"].isin(config.SMOG_SEASON_MONTHS).astype(int)

print(df.shape)
print(f"Range: {df['timestamp'].min()} -> {df['timestamp'].max()}")
df.head()


## Load raw merged snapshot (no Hopsworks, no engineered features)

In [ ]:
# Data already loaded in the PREVIOUS code cell (imports + load).
# Do NOT run any old cell that says: df = load_raw_snapshot()
# If df is missing, re-run the first code cell only.
print("df ready:", "df" in dir())
print(df.shape if "df" in dir() else "Re-run the first code cell")


## Structure check

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum()[df.isnull().sum() > 0])
print('\nDuplicate timestamps:', df['timestamp'].duplicated().sum())
df.describe().T

## Univariate analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['aqi'], kde=True, ax=axes[0])
axes[0].set_title('AQI Distribution')
sns.boxplot(x=df['aqi'], ax=axes[1])
axes[1].set_title('AQI Boxplot (outlier check)')
plt.tight_layout(); plt.show()

print('AQI skewness:', df['aqi'].skew())
print('AQI kurtosis:', df['aqi'].kurtosis())

In [ ]:
pollutants = [c for c in ['pm2_5', 'pm10', 'co', 'no', 'no2', 'o3', 'so2', 'nh3'] if c in df.columns]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, pollutants):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'{col} (skew={df[col].skew():.2f})')
for ax in axes.flat[len(pollutants):]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()

print('Pollutant skewness:')
print(df[pollutants].skew().sort_values(ascending=False))

In [ ]:
plt.figure(figsize=(16, 4))
plt.plot(df['timestamp'], df['aqi'], linewidth=0.5)
plt.title('AQI Over Time — Lahore (raw)')
plt.xlabel('Date'); plt.ylabel('AQI')
plt.show()

# Yearly means — trend check (known multi-year decline)
yearly = df.set_index('timestamp')['aqi'].resample('YE').mean()
print('Yearly mean AQI:')
print(yearly)

## Bivariate analysis — AQI vs weather variables

In [ ]:
weather_cols = [c for c in ['temperature', 'humidity', 'wind_speed', 'pressure'] if c in df.columns]
fig, axes = plt.subplots(1, len(weather_cols), figsize=(18, 4))
for ax, col in zip(axes, weather_cols):
    sns.scatterplot(x=df[col], y=df['aqi'], alpha=0.15, ax=ax, s=8)
    ax.set_title(f'AQI vs {col} (r={df[col].corr(df["aqi"]):.2f})')
plt.tight_layout(); plt.show()

**Note (Lahore-specific):** expect a negative relationship between `wind_speed` and AQI
(higher wind disperses pollution) and a positive relationship between `humidity` and AQI
during smog season (secondary particle formation).

## Multivariate analysis — correlation heatmap (raw columns only)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['month'], errors='ignore')
plt.figure(figsize=(12, 9))
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap — Raw Columns')
plt.show()

print('Top correlations with aqi:')
print(numeric_df.corr()['aqi'].sort_values(ascending=False))

## Time-series: stationarity, ACF/PACF, seasonal decomposition
These findings justify which lag/rolling features to keep in feature engineering.

In [ ]:
result = adfuller(df['aqi'].dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print('=> Stationary (reject H0)' if result[1] < 0.05 else '=> Non-stationary (fail to reject H0) — consider differencing / delta targets')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df['aqi'].dropna(), lags=168, ax=axes[0])
plot_pacf(df['aqi'].dropna(), lags=72, ax=axes[1], method='ywm')
axes[0].set_title('ACF (lags up to 168h)')
axes[1].set_title('PACF (lags up to 72h)')
plt.tight_layout(); plt.show()
# Significant spikes justify WHICH lag features to keep (e.g. lag 24 / 168).

In [ ]:
ts = df.set_index('timestamp')['aqi'].asfreq('h').interpolate()
decomposition = seasonal_decompose(ts, model='additive', period=24)
fig = decomposition.plot()
fig.set_size_inches(14, 8)
plt.tight_layout(); plt.show()

## Smog season vs. normal season

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x=df['is_smog_season'].map({0: 'Normal', 1: 'Smog (Oct-Jan)'}), y=df['aqi'])
plt.title('AQI: Smog Season vs. Normal Season')
plt.xlabel(''); plt.ylabel('AQI')
plt.show()

print(df.groupby('is_smog_season')['aqi'].describe())

## Findings for FE (filled from raw snapshot — approved for FE)

Verified on `data/raw/aqi_raw_merged.parquet` (46,340 rows,
2020-11-27 00:00:00 -> 2026-08-11 06:00:00).

| Topic | Finding |
|-------|---------|
| Skewness (aqi + pollutants) | AQI skew=1.91. Pollutants (desc): no=3.92, nh3=3.01, so2=2.46, no2=2.26, co=1.99, pm2_5=1.41, pm10=1.29, o3=1.26. All right-skewed -> log1p candidates for pollutants. |
| Outliers | AQI IQR upper fence ~478; 5,876 rows (12.7%) above it (smog extremes). Keep for now; prefer robust trees + delta target over aggressive clipping. |
| Correlations (aqi vs weather/pollutants) | Top: pm2_5=0.99, pm10=0.96, co=0.79, no=0.56, no2=0.55. Weather: temp=-0.55, pressure=0.51, humidity=0.29, wind=-0.24. |
| ADF stationarity | ADF p=3.80e-24 -> reject unit root (statistically stationary) but strong short-term persistence remains. |
| ACF/PACF — which lags matter | ACF: lag1=0.95, lag3=0.81, lag6=0.68, lag24=0.65, lag48=0.53, lag72=0.50, lag168=0.43. PACF: lag1=0.95, lag2=-0.25, lag5=0.19, lag25=-0.20. Keep lags 1,3,6,24,168; rolling 3/6/24 justified. |
| Trend / seasonality | Clear daily seasonality (ACF@24). Multi-year downward trend in yearly means (see below) -> **delta targets** required. |
| Smog vs normal | Normal mean AQI=182.5; Smog (Oct-Jan) mean=399.1 (~2.2x). Keep `is_smog_season` + stratified eval. |
| Yearly mean trend | 2020=438.8; 2021=263.9; 2022=277.1; 2023=280.1; 2024=251.6; 2025=202.9; 2026=167.8 |

**Proposed FE actions (locked from this table):**
1. `log1p` transform on right-skewed pollutants (`no`,`nh3`,`so2`,`no2`,`co`,`pm2_5`,`pm10`,`o3`)
2. Keep lag set {1,3,6,24,168} and rolling windows {3,6,24} (ACF/PACF justified)
3. Keep weather interactions (wind/humidity with pm2_5) given signed correlations
4. Primary targets: `aqi_delta_{24,48,72}h = aqi_target - aqi` (absolute targets kept for metrics)
5. Scale tabular features with StandardScaler inside Ridge pipeline; trees unscaled


In [ ]:
# Auto-summary helpers — paste key numbers into the markdown table above
print('=== Auto findings snapshot ===')
print('AQI skew:', round(df['aqi'].skew(), 3))
print('AQI mean/median/std:', round(df['aqi'].mean(), 1), round(df['aqi'].median(), 1), round(df['aqi'].std(), 1))
iqr = df['aqi'].quantile(0.75) - df['aqi'].quantile(0.25)
outlier_hi = df['aqi'].quantile(0.75) + 1.5 * iqr
print(f'AQI IQR outliers above {outlier_hi:.0f}:', int((df['aqi'] > outlier_hi).sum()))
print('\nPollutant skew:')
print(df[pollutants].skew().round(3).sort_values(ascending=False))
print('\nCorr with aqi:')
print(numeric_df.corr()['aqi'].sort_values(ascending=False).round(3))
print('\nADF p-value:', round(adfuller(df['aqi'].dropna())[1], 6))
print('\nYearly mean AQI:')
print(df.set_index('timestamp')['aqi'].resample('YE').mean().round(1))
print('\nSmog vs normal mean AQI:')
print(df.groupby('is_smog_season')['aqi'].mean().round(1))